In [1]:
import pandas as pd
import numpy as np

## Phase 2: Data Cleaning

This notebook cleans the raw Olist CSVs based on issues identified during 
inspection in `01_data_loading.ipynb`. Key fixes: datetime conversions, 
zip code type corrections, and handling missing values in `products` and `orders`.

### Step 1: Load raw data
Reloading CSVs fresh (not reusing the loading notebook) so this notebook 
runs independently, top to bottom.

In [2]:
import pandas as pd
import numpy as np

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

### Step 2: Fix datetime columns — `orders`

All date columns loaded as `object` (text) instead of proper datetime type. 
Converting with `pd.to_datetime()` so date arithmetic (e.g., delivery delay) 
works correctly in later analysis.

In [3]:
date_columns_orders = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns_orders:
    orders[col] = pd.to_datetime(orders[col])

In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


### Step 3: Fix datetime columns — `order_reviews`

Same issue as `orders`: `review_creation_date` and `review_answer_timestamp` 
loaded as text.

In [5]:
date_columns_order_reviews = [
    'review_creation_date',
    'review_answer_timestamp'
]

for col in date_columns_order_reviews:
    order_reviews[col] = pd.to_datetime(order_reviews[col])

In [6]:
order_reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   order_id                 99224 non-null  object        
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  object        
 4   review_comment_message   40977 non-null  object        
 5   review_creation_date     99224 non-null  datetime64[ns]
 6   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 5.3+ MB


### Step 4: Fix datetime column — `order_items`

`shipping_limit_date` loaded as text. Only one column here, so a direct 
conversion is used instead of a loop.

In [7]:
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])

In [8]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  object        
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  object        
 3   seller_id            112650 non-null  object        
 4   shipping_limit_date  112650 non-null  datetime64[ns]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(3)
memory usage: 6.0+ MB


### Step 5: Fix zip code columns — `customers`, `sellers`, `geolocation`

`customer_zip_code_prefix`, `seller_zip_code_prefix`, and 
`geolocation_zip_code_prefix` loaded as `int64`. Zip codes are identifiers, 
not quantities meant for calculation, so storing them as integers risks 
silently dropping leading zeros — which would break joins on these columns 
later. Converting all three to string type to preserve the original values.

In [9]:
customers['customer_zip_code_prefix'] = customers['customer_zip_code_prefix'].astype(str)
sellers['seller_zip_code_prefix'] = sellers['seller_zip_code_prefix'].astype(str)
geolocation['geolocation_zip_code_prefix'] = geolocation['geolocation_zip_code_prefix'].astype(str)

In [10]:
print(customers['customer_zip_code_prefix'].dtype)
print(sellers['seller_zip_code_prefix'].dtype)
print(geolocation['geolocation_zip_code_prefix'].dtype)

object
object
object


### Step 6: Investigate — products missing category

Before deciding how to handle the 610 products missing `product_category_name` 
(drop vs. label as "unknown"), first check how much actual revenue these 
products represent. This determines whether the missing category issue is 
negligible or significant enough to require careful handling rather than a 
quick fix.

In [11]:
missing_category_products = products[products['product_category_name'].isnull()]['product_id']
revenue_missing = order_items[order_items['product_id'].isin(missing_category_products)]['price'].sum()
total_revenue = order_items['price'].sum()

print(f"Revenue from products missing category: R$ {revenue_missing:,.2f}")
print(f"Total revenue: R$ {total_revenue:,.2f}")
print(f"Percentage: {(revenue_missing / total_revenue) * 100:.2f}%")

Revenue from products missing category: R$ 179,535.28
Total revenue: R$ 13,591,643.70
Percentage: 1.32%


### Step 7: Handle missing product category

Products missing category represent 1.32% of total revenue (R$179,535 of 
R$13,591,643) — too small to be a major driver, but significant enough that 
dropping them would understate revenue and create a mismatch with source 
data. Decision: keep these products, label missing category as "unknown" 
rather than drop, to preserve revenue accuracy and keep the gap explainable.

In [12]:
products['product_category_name'] = products['product_category_name'].fillna('unknown')

In [13]:
products['product_category_name'].isnull().sum()

np.int64(0)

### Step 8: Investigate — missing delivery dates in `orders`

Unlike the product category issue, missing delivery dates are likely NOT a 
data quality problem — an order that was cancelled or never delivered should 
correctly have a null `order_delivered_customer_date`. Before deciding how 
to handle these nulls, verify this hypothesis by checking the `order_status` 
of orders with missing delivery dates. If missing dates correlate with 
non-delivered statuses (cancelled, unavailable, etc.), the nulls are accurate 
and should be left as-is rather than filled or dropped.

In [14]:
orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

### Step 9: Decision — missing delivery dates

Investigation confirms missing `order_delivered_customer_date` values 
correlate almost entirely with non-delivered order statuses (shipped, 
canceled, unavailable, invoiced, processing, created, approved) — these 
nulls are accurate, not data errors, and will be left as-is.

One anomaly found: 8 orders marked `delivered` still have a null delivery 
date — likely a logging inconsistency at the source. This affects 0.008% 
of orders and is treated as a negligible, noted limitation rather than 
corrected, since we cannot know the true delivery date.

Decision: no changes made to `order_delivered_customer_date`, 
`order_delivered_carrier_date`, or `order_approved_at`. Nulls are preserved 
as meaningful signals of order status, not treated as missing data to fill.

### Step 10: Save cleaned data

Exporting all cleaned DataFrames to `data/cleaned/` as CSV files. These 
cleaned files will be the source for Phase 3 (re-importing into MySQL) and 
Phase 2's remaining feature engineering step, rather than re-running this 
cleaning notebook every time.

In [15]:
orders.to_csv('../data/cleaned/orders_cleaned.csv', index=False)
customers.to_csv('../data/cleaned/customers_cleaned.csv', index=False)
sellers.to_csv('../data/cleaned/sellers_cleaned.csv', index=False)
products.to_csv('../data/cleaned/products_cleaned.csv', index=False)
order_items.to_csv('../data/cleaned/order_items_cleaned.csv', index=False)
order_payments.to_csv('../data/cleaned/order_payments_cleaned.csv', index=False)
order_reviews.to_csv('../data/cleaned/order_reviews_cleaned.csv', index=False)
geolocation.to_csv('../data/cleaned/geolocation_cleaned.csv', index=False)
category_translation.to_csv('../data/cleaned/category_translation_cleaned.csv', index=False)